# Tahoe-100M Data Loading and DataLoader Pipeline

This notebook streams Tahoe-100M from HuggingFace, organizes records as **cell line -> perturbation -> expression matrix**, and prepares PyTorch DataLoaders using an **80/10/10 perturbation-level split** for predicting unseen perturbations on known cell lines.

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
import anndata as ad
import torch
from datasets import load_dataset
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

In [ ]:
DATASET_NAME = "vevotx/Tahoe-100M"
SAMPLE_SIZE = 200_000  # Set to None to stream the full dataset (very large).
REPORT_EVERY = 25_000


def build_cellline_perturbation_index(streaming_ds, gene_vocab, sample_size=None, report_every=None):
    sorted_vocab_items = sorted(gene_vocab.items())
    token_ids, gene_names = zip(*sorted_vocab_items)
    token_id_to_col_idx = {token_id: idx for idx, token_id in enumerate(token_ids)}

    grouped_buffers = defaultdict(
        lambda: {"data": [], "indices": [], "indptr": [0], "obs": []}
    )

    for i, record in enumerate(streaming_ds):
        if sample_size is not None and i >= sample_size:
            break

        genes = record["genes"]
        expressions = record["expressions"]

        # The tutorial notebook drops a leading sentinel value when expression starts negative.
        if expressions and expressions[0] < 0:
            genes = genes[1:]
            expressions = expressions[1:]

        row_indices = []
        row_values = []
        for gene_token, expr_value in zip(genes, expressions):
            col_idx = token_id_to_col_idx.get(gene_token)
            if col_idx is not None:
                row_indices.append(col_idx)
                row_values.append(float(expr_value))

        key = (record["cell_line_id"], record["drug"])
        buffer = grouped_buffers[key]
        buffer["indices"].extend(row_indices)
        buffer["data"].extend(row_values)
        buffer["indptr"].append(len(buffer["data"]))
        buffer["obs"].append(
            {k: v for k, v in record.items() if k not in ("genes", "expressions")}
        )

        if report_every and (i + 1) % report_every == 0:
            print(f"Streamed {i + 1:,} cells. Current groups: {len(grouped_buffers):,}")

    data_by_cell_and_drug = defaultdict(dict)
    var_index = pd.Index(gene_names, name="ensembl_id")

    for (cell_line_id, perturbation), buffer in grouped_buffers.items():
        x_matrix = csr_matrix(
            (
                np.asarray(buffer["data"], dtype=np.float32),
                np.asarray(buffer["indices"], dtype=np.int32),
                np.asarray(buffer["indptr"], dtype=np.int64),
            ),
            shape=(len(buffer["obs"]), len(gene_names)),
            dtype=np.float32,
        )

        obs_df = pd.DataFrame(buffer["obs"])
        pair_adata = ad.AnnData(X=x_matrix, obs=obs_df)
        pair_adata.var.index = var_index
        data_by_cell_and_drug[cell_line_id][perturbation] = pair_adata

    return data_by_cell_and_drug, list(gene_names)


tahoe_100m_stream = load_dataset(DATASET_NAME, streaming=True, split="train")
gene_metadata = load_dataset(DATASET_NAME, name="gene_metadata", split="train")
gene_vocab = {entry["token_id"]: entry["ensembl_id"] for entry in gene_metadata}

data_by_cell_and_drug, gene_names = build_cellline_perturbation_index(
    tahoe_100m_stream,
    gene_vocab,
    sample_size=SAMPLE_SIZE,
    report_every=REPORT_EVERY,
)

print(f"Gene features: {len(gene_names):,}")
print(f"Cell lines observed: {len(data_by_cell_and_drug):,}")

In [ ]:
def summarize_hierarchy(data_by_cell_and_drug):
    rows = []
    for cell_line_id, perturbation_map in data_by_cell_and_drug.items():
        for perturbation, pair_adata in perturbation_map.items():
            rows.append(
                {
                    "cell_line_id": cell_line_id,
                    "perturbation": perturbation,
                    "n_cells": int(pair_adata.n_obs),
                    "n_genes": int(pair_adata.n_vars),
                }
            )
    return pd.DataFrame(rows)


def expression_df_for_pair(data_by_cell_and_drug, cell_line_id, perturbation):
    pair_adata = data_by_cell_and_drug[cell_line_id][perturbation]
    if "BARCODE_SUB_LIB_ID" in pair_adata.obs.columns:
        row_index = pair_adata.obs["BARCODE_SUB_LIB_ID"].astype(str)
    else:
        row_index = pair_adata.obs.index.astype(str)

    expr_df = pd.DataFrame.sparse.from_spmatrix(
        pair_adata.X,
        index=row_index,
        columns=pair_adata.var_names,
    )
    return expr_df


pair_summary_df = summarize_hierarchy(data_by_cell_and_drug)

if pair_summary_df.empty:
    print("No records were loaded. Increase SAMPLE_SIZE or check dataset access.")
else:
    cell_line_summary_df = (
        pair_summary_df.groupby("cell_line_id", as_index=False)
        .agg(
            n_perturbations=("perturbation", "nunique"),
            n_cells=("n_cells", "sum"),
        )
        .sort_values(["n_perturbations", "n_cells"], ascending=False)
    )

    print(f"Cell lines indexed: {pair_summary_df['cell_line_id'].nunique():,}")
    print(f"Unique perturbations observed: {pair_summary_df['perturbation'].nunique():,}")
    print(f"Cell-line/perturbation groups: {len(pair_summary_df):,}")

    display(cell_line_summary_df.head(10))
    display(pair_summary_df.sort_values("n_cells", ascending=False).head(10))

    example_cell_line, example_perturbation = pair_summary_df.loc[
        0, ["cell_line_id", "perturbation"]
    ]
    example_expr_df = expression_df_for_pair(
        data_by_cell_and_drug,
        example_cell_line,
        example_perturbation,
    )

    print(
        f"Example pair ({example_cell_line}, {example_perturbation}) expression matrix shape: "
        f"{example_expr_df.shape}"
    )
    display(example_expr_df.iloc[:3, :10])

In [ ]:
sample_metadata = load_dataset(DATASET_NAME, name="sample_metadata", split="train")
all_perturbations = sorted(set(sample_metadata["drug"]))

rng = np.random.default_rng(42)
shuffled_perturbations = np.array(all_perturbations, dtype=object)
rng.shuffle(shuffled_perturbations)

n_total = len(shuffled_perturbations)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_perturbations = set(shuffled_perturbations[:n_train].tolist())
val_perturbations = set(shuffled_perturbations[n_train : n_train + n_val].tolist())
test_perturbations = set(shuffled_perturbations[n_train + n_val :].tolist())

print(f"Total perturbations from metadata: {n_total:,}")
print(f"Train perturbations: {len(train_perturbations):,}")
print(f"Val perturbations: {len(val_perturbations):,}")
print(f"Test perturbations: {len(test_perturbations):,}")

# Perturbations present in the currently streamed subset.
observed_perturbations = {
    perturbation
    for perturbation_map in data_by_cell_and_drug.values()
    for perturbation in perturbation_map.keys()
}
print(f"Perturbations observed in streamed subset: {len(observed_perturbations):,}")

In [ ]:
class PerturbationDataset(Dataset):
    def __init__(
        self,
        data_by_cell_and_drug,
        allowed_perturbations,
        cell_line_to_idx,
        perturbation_to_idx,
    ):
        self.data_by_cell_and_drug = data_by_cell_and_drug
        self.allowed_perturbations = set(allowed_perturbations)
        self.cell_line_to_idx = cell_line_to_idx
        self.perturbation_to_idx = perturbation_to_idx

        self.samples = []
        for cell_line_id, perturbation_map in self.data_by_cell_and_drug.items():
            for perturbation, pair_adata in perturbation_map.items():
                if perturbation not in self.allowed_perturbations:
                    continue
                for row_idx in range(pair_adata.n_obs):
                    self.samples.append((cell_line_id, perturbation, row_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cell_line_id, perturbation, row_idx = self.samples[idx]
        pair_adata = self.data_by_cell_and_drug[cell_line_id][perturbation]

        sparse_row = pair_adata.X[row_idx]
        if hasattr(sparse_row, "toarray"):
            expression_np = sparse_row.toarray().ravel().astype(np.float32, copy=False)
        else:
            expression_np = np.asarray(sparse_row, dtype=np.float32).ravel()

        return {
            "expression": torch.from_numpy(expression_np),
            "cell_line_idx": torch.tensor(self.cell_line_to_idx[cell_line_id], dtype=torch.long),
            "perturbation_idx": torch.tensor(self.perturbation_to_idx[perturbation], dtype=torch.long),
            "cell_line_id": cell_line_id,
            "perturbation": perturbation,
        }


all_cell_lines = sorted(data_by_cell_and_drug.keys())
cell_line_to_idx = {cell_line_id: i for i, cell_line_id in enumerate(all_cell_lines)}
perturbation_to_idx = {
    perturbation: i for i, perturbation in enumerate(sorted(all_perturbations))
}

# Restrict each split to perturbations that are present in the streamed subset.
train_perturbations_observed = train_perturbations & observed_perturbations
val_perturbations_observed = val_perturbations & observed_perturbations
test_perturbations_observed = test_perturbations & observed_perturbations

train_ds = PerturbationDataset(
    data_by_cell_and_drug,
    train_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)
val_ds = PerturbationDataset(
    data_by_cell_and_drug,
    val_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)
test_ds = PerturbationDataset(
    data_by_cell_and_drug,
    test_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
def count_cells_in_split(data_by_cell_and_drug, perturbation_set):
    total_cells = 0
    for perturbation_map in data_by_cell_and_drug.values():
        for perturbation, pair_adata in perturbation_map.items():
            if perturbation in perturbation_set:
                total_cells += int(pair_adata.n_obs)
    return total_cells


print("=== Perturbation split (global metadata) ===")
print(f"Train perturbations: {len(train_perturbations):,}")
print(f"Val perturbations:   {len(val_perturbations):,}")
print(f"Test perturbations:  {len(test_perturbations):,}")

print("\n=== Perturbations represented in streamed subset ===")
print(f"Train perturbations observed: {len(train_perturbations_observed):,}")
print(f"Val perturbations observed:   {len(val_perturbations_observed):,}")
print(f"Test perturbations observed:  {len(test_perturbations_observed):,}")

print("\n=== Cell counts represented in streamed subset ===")
print(f"Train cells: {count_cells_in_split(data_by_cell_and_drug, train_perturbations_observed):,}")
print(f"Val cells:   {count_cells_in_split(data_by_cell_and_drug, val_perturbations_observed):,}")
print(f"Test cells:  {count_cells_in_split(data_by_cell_and_drug, test_perturbations_observed):,}")

print("\n=== Dataset lengths ===")
print(f"len(train_ds): {len(train_ds):,}")
print(f"len(val_ds):   {len(val_ds):,}")
print(f"len(test_ds):  {len(test_ds):,}")

if len(train_ds) == 0:
    print("\nTrain dataset is empty. Increase SAMPLE_SIZE or set SAMPLE_SIZE=None.")
else:
    train_batch = next(iter(train_loader))
    print("\n=== One train batch ===")
    print("expression shape:", tuple(train_batch["expression"].shape))
    print("cell_line_idx shape:", tuple(train_batch["cell_line_idx"].shape))
    print("perturbation_idx shape:", tuple(train_batch["perturbation_idx"].shape))
    print("example cell_line_ids:", train_batch["cell_line_id"][:5])
    print("example perturbations:", train_batch["perturbation"][:5])